In [2]:
!pip install gurobipy

   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.4 MB 991.0 kB/s eta 0:00:12
    --------------------------------------- 0.2/11.4 MB 2.6 MB/s eta 0:00:05
   - -------------------------------------- 0.3/11.4 MB 2.9 MB/s eta 0:00:04
   - -------------------------------------- 0.5/11.4 MB 3.1 MB/s eta 0:00:04
   -- ------------------------------------- 0.7/11.4 MB 2.9 MB/s eta 0:00:04
   -- ------------------------------------- 0.8/11.4 MB 2.8 MB/s eta 0:00:04
   --- ------------------------------------ 0.9/11.4 MB 3.0 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.4 MB 3.0 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.4 MB 3.0 MB/s eta 0:00:04
   ---- ----------------------------------- 1.2/11.4 MB 2.7 MB/s eta 0:00:04
   ---- ----------------------------------- 1.3/11.4 MB 2.7 MB/s eta 0:00:04
   ----- ---------------------------------- 1.5/11.4 MB 2.8 MB/s eta 0:00:04
   -


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Huypz\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## 1.Set Parameters

In [4]:
I = list(range(1,21)) # Employees: 1, 2, ..., 20
J = list(range(1,8)) # Days: 1, 2, ..., 7
K = list(range(1,4)) # Shifts: 1, 2, 3 # 1: Morning, 2: Afternoon, 3: Night

In [13]:
# Rules: 5 first employees are senior, remaining 15 are junior
Seniors = set(range(1,6)) # Employees 1-5 are seniors
### 1. Staffs per shift (d_jk) based on README
demand = {
    (1, 1): 6, (1, 2): 4, (1, 3): 3,
    (2, 1): 5, (2, 2): 5, (2, 3): 5,
    (3, 1): 6, (3, 2): 4, (3, 3): 4,
    (4, 1): 5, (4, 2): 5, (4, 3): 5,
    (5, 1): 5, (5, 2): 5, (5, 3): 3,
    (6, 1): 6, (6, 2): 5, (6, 3): 4, # Saturday needs a total of 15 people
    (7, 1): 5, (7, 2): 5, (7, 3): 4  # Sunday needs a total of 14 people
}

## 2. Model Activating

In [6]:
m = gp.Model("Shift_Scheduling")

Restricted license - for non-production use only - expires 2027-11-29


## 3.Decision Variables

In [7]:
# x[i, j, k] = 1 employee i works on day j in shift k, otherwise = 0
x = m.addVars(I, J, K, vtype=GRB.BINARY, name="x")

### 3.1 Support Variables for Load Balancing

In [14]:
W_max = m.addVar(vtype=GRB.INTEGER, name="W_max")
W_min = m.addVar(vtype=GRB.INTEGER, name="W_min")

### 3.2 Support Variable for Weekend

In [15]:
# if employee i work at weekend
y = m.addVars(I, vtype=GRB.BINARY, name="y")

## 4. Objective Function

In [16]:
# Required number of employees for each shift (variable d_jk/Penalty Cost)
# The weight "5" means: Giving employees weekends off is 5 times more important than balancing the shift differences among employees.
m.setObjective((W_max - W_min) + 5 * gp.quicksum(y[i] for i in I), GRB.MINIMIZE)

### 5.Normal Constraints

In [18]:
for j in J:
    for k in K:
        m.addConstr(gp.quicksum(x[i, j, k] for i in I) == demand[j, k]) # Fulfill Demand
        
for i in I:
    for j in J:
        m.addConstr(gp.quicksum(x[i, j, k] for k in K) <= 1) # Max 1 shift per day
    m.addConstr(gp.quicksum(x[i, j, k] for j in J for k in K) <= 5) # Max 5 shifts per week

for i in I:
    for j in range(1, 7):
        m.addConstr(x[i, j, 3] + x[i, j+1, 1] <= 1) # Rest period (Night -> Morning)

### 6. New Constraints for Optimization

In [19]:
# [Optimization 1] Load Balancing: Minimize the difference between the most and least loaded employees (W_max - W_min)
for i in I:
    w_i = gp.quicksum(x[i, j, k] for j in J for k in K)
    m.addConstr(w_i <= W_max, name=f"MaxLoad_{i}")
    m.addConstr(w_i >= W_min, name=f"MinLoad_{i}")

# [Optimization 2] Skill-mix: Each shift must have at least 1 Senior (employee 1-5)
for j in J:
    for k in K:
        m.addConstr(gp.quicksum(x[i, j, k] for i in Seniors) >= 1, name=f"SeniorReq_{j}_{k}")

# [Optimization 3] Weekend Rest (Soft Constraint)
# If an employee works on Saturday and Sunday, then y[i] must be 1. The objective function will try to minimize y[i] as much as possible.
for i in I:
    m.addConstr(gp.quicksum(x[i, 6, k] for k in K) + gp.quicksum(x[i, 7, k] for k in K) - 1 <= y[i], name=f"Weekend_{i}")

### 7. Extract Results and Export CSV

In [20]:
# 7. Run and Export Results
m.optimize()

if m.Status == GRB.OPTIMAL:
    print(f"\n✅ Solved successfully! Highest shift difference: {int(W_max.X - W_min.X)} shifts.")
    print(f"Number of employees required to work both Saturday and Sunday: {int(sum(y[i].X for i in I))} people.")

    records = []
    for i in I:
        for j in J:
            assigned = False
            for k in K:
                if x[i, j, k].X > 0.5:
                    records.append({'Staff': i, 'Role': 'Senior' if i in Seniors else 'Junior', 'Day': j, 'Shift': k, 'Status': 'Assigned'})
                    assigned = True
                    break
            if not assigned:
                records.append({'Staff': i, 'Role': 'Senior' if i in Seniors else 'Junior', 'Day': j, 'Shift': None, 'Status': 'Leave'})
                
    df_out = pd.DataFrame(records)
    df_out.to_csv("master_optimized_staff_assignment.csv", index=False)
    print("Optimal staff assignment saved to 'master_optimized_staff_assignment.csv'.")
else:
    print("No feasible solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 984 rows, 442 columns and 5625 nonzeros (Min)
Model fingerprint: 0xad21b04c
Model has 22 linear objective coefficients
Variable types: 0 continuous, 442 integer (440 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 5e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]

MIP start from previous solve did not produce a new incumbent solution
MIP start from previous solve violates constraint SeniorReq_1_2 by 1.000000000

Found heuristic solution: objective 61.0000000
Presolve removed 602 rows and 0 columns
Presolve time: 0.01s
Presolved: 382 rows, 442 columns, 2625 nonzeros
Variable types: 0 continuous, 442 integer (440 binary)

Root relaxation: objective 4.500000e

### 7.Pivot Table

In [21]:
# 1. Create a new column 'Schedule' to indicate the shift or leave status for each record
# If (Leave) then fill with 'O' (Off), if working then fill with 'Shift 1', 'Shift 2', 'Shift 3'
df_out['Schedule'] = df_out.apply(lambda row: 'O' if row['Status'] == 'Leave' else f"Shift {int(row['Shift'])}", axis=1)

# 2. Rotate the data (Pivot)
# - Row (index): Employee name and role (Role)
# - Column (columns): Work day (Day 1 to 7)
# - Value (values): Shift assigned (Schedule)
pivot_df = df_out.pivot(index=['Staff', 'Role'], columns='Day', values='Schedule')

# 3. Fill any empty cells (if any) with 'O' to prevent display errors
pivot_df = pivot_df.fillna('O')

# 4. Display the pivot table in the Colab output
print("=== Table of Staff Assignments ===")
display(pivot_df)

# 5. Save to Excel file for management or staff review
pivot_df.to_excel("staff_assignment_pivot.xlsx")
print("\nOptimal shift schedule saved to 'staff_assignment_pivot.xlsx'")

=== Table of Staff Assignments ===


,Day,1,2,3,4,5,6,7
Staff,Role,,,,,,,
1,Senior,O,Shift 2,Shift 1,Shift 1,Shift 3,Shift 3,O
2,Senior,Shift 3,O,O,Shift 3,Shift 2,Shift 1,Shift 1
3,Senior,Shift 2,O,Shift 3,Shift 2,Shift 1,O,Shift 3
4,Senior,Shift 1,Shift 1,Shift 1,O,Shift 2,Shift 2,O
5,Senior,O,Shift 3,Shift 2,O,O,Shift 3,Shift 2
6,Junior,O,Shift 1,Shift 1,O,Shift 1,Shift 1,Shift 1
7,Junior,Shift 1,Shift 2,Shift 2,Shift 1,O,Shift 1,O
8,Junior,Shift 1,Shift 2,O,Shift 1,Shift 2,O,Shift 2
9,Junior,O,Shift 3,Shift 2,Shift 3,O,Shift 2,Shift 1



Optimal shift schedule saved to 'staff_assignment_pivot.xlsx'
